In [0]:
%run ./00_Config_and_Logging

In [0]:
# ==========================================
# 03_Silver_to_Gold: configuration, reference data, WHO thresholds
# Gold tables are Unity Catalog tables in workspace.gold so Power BI / Tableau
# can query them through the SQL warehouse (path-only Delta files in volumes cannot be).
# ==========================================
import os
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import col

silver_quarantine_path = f"{os.path.dirname(silver_delta_path)}/silver_quarantine"

GOLD = "workspace.gold"
T_LOC, T_DATE, T_FACT, T_WM = (f"{GOLD}.{n}" for n in
    ["dim_location", "dim_date", "fact_concentrations", "gold_watermark"])

# City -> country, grouped by region (descriptive attributes for dim_location)
REGIONS = {
 "South Asia": {"Lahore": "Pakistan", "Delhi": "India", "Karachi": "Pakistan", "Mumbai": "India",
    "Kolkata": "India", "Bangalore": "India", "Hyderabad": "India", "Ahmedabad": "India",
    "Chennai": "India", "Dhaka": "Bangladesh", "Colombo": "Sri Lanka", "Kathmandu": "Nepal",
    "Islamabad": "Pakistan", "Kabul": "Afghanistan"},
 "East & Southeast Asia": {"Beijing": "China", "Shanghai": "China", "Guangzhou": "China",
    "Shenzhen": "China", "Wuhan": "China", "Chengdu": "China", "Xian": "China", "Tianjin": "China",
    "Hong_Kong": "Hong Kong", "Tokyo": "Japan", "Osaka": "Japan", "Nagoya": "Japan",
    "Seoul": "South Korea", "Taipei": "Taiwan", "Bangkok": "Thailand", "Jakarta": "Indonesia",
    "Manila": "Philippines", "Ho_Chi_Minh_City": "Vietnam", "Hanoi": "Vietnam",
    "Yangon": "Myanmar", "Kuala_Lumpur": "Malaysia", "Singapore": "Singapore"},
 "Middle East": {"Tehran": "Iran", "Baghdad": "Iraq", "Riyadh": "Saudi Arabia", "Dubai": "UAE",
    "Ankara": "Turkey", "Istanbul": "Turkey", "Amman": "Jordan", "Beirut": "Lebanon"},
 "Africa": {"Cairo": "Egypt", "Lagos": "Nigeria", "Kinshasa": "DR Congo",
    "Johannesburg": "South Africa", "Cape_Town": "South Africa", "Nairobi": "Kenya",
    "Addis_Ababa": "Ethiopia", "Casablanca": "Morocco", "Algiers": "Algeria", "Tunis": "Tunisia",
    "Accra": "Ghana", "Dakar": "Senegal", "Abidjan": "Ivory Coast", "Kampala": "Uganda",
    "Dar_es_Salaam": "Tanzania", "Luanda": "Angola", "Harare": "Zimbabwe"},
 "Europe": {"London": "United Kingdom", "Paris": "France", "Berlin": "Germany", "Madrid": "Spain",
    "Rome": "Italy", "Milan": "Italy", "Barcelona": "Spain", "Lisbon": "Portugal",
    "Amsterdam": "Netherlands", "Brussels": "Belgium", "Vienna": "Austria", "Warsaw": "Poland",
    "Prague": "Czechia", "Budapest": "Hungary", "Athens": "Greece", "Bucharest": "Romania",
    "Kyiv": "Ukraine", "Moscow": "Russia", "Stockholm": "Sweden", "Oslo": "Norway",
    "Helsinki": "Finland", "Dublin": "Ireland"},
 "North America": {"New_York": "USA", "Los_Angeles": "USA", "Chicago": "USA", "Houston": "USA",
    "Miami": "USA", "Toronto": "Canada", "Vancouver": "Canada", "Mexico_City": "Mexico"},
 "South America": {"Sao_Paulo": "Brazil", "Rio_de_Janeiro": "Brazil", "Buenos_Aires": "Argentina",
    "Lima": "Peru", "Bogota": "Colombia", "Santiago": "Chile"},
 "Oceania": {"Sydney": "Australia", "Melbourne": "Australia", "Auckland": "New Zealand"},
}
LOCATION_REF = spark.createDataFrame(
    [(c, country, region) for region, d in REGIONS.items() for c, country in d.items()],
    "city string, country string, region string")

# WHO 2021 air-quality guideline values, 24-hour mean, in ug/m3 (CO: 4 mg/m3).
# The WHO publishes no CO2 guideline, so CO2 is analysed as a trend, not against a threshold.
WHO_24H = {"pm2_5": 15.0, "pm10": 45.0, "no2": 25.0, "co": 4000.0}
print("Reference cities:", LOCATION_REF.count())

In [0]:
# ==========================================
# Gold data model (star schema). Grain of the fact table: one row per city per day.
# Informational PK/FK constraints let BI tools detect the one-to-many relationships.
# The fact table uses Liquid Clustering on its two keys (read optimisation).
# ==========================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_LOC} (
  location_key INT NOT NULL, city STRING NOT NULL, country STRING, region STRING,
  latitude DOUBLE, longitude DOUBLE, elevation DOUBLE, timezone STRING,
  hemisphere STRING, climate_band STRING, load_timestamp TIMESTAMP NOT NULL)""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_DATE} (
  date_key INT NOT NULL, full_date DATE NOT NULL, year INT, quarter INT, month INT,
  month_name STRING, day INT, day_of_week INT, day_name STRING, is_weekend BOOLEAN,
  season_northern_hemisphere STRING, load_timestamp TIMESTAMP NOT NULL)""")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_FACT} (
  location_key INT NOT NULL, date_key INT NOT NULL, hours_measured INT, is_complete_day BOOLEAN,
  avg_carbon_monoxide_ugm3 DOUBLE, max_carbon_monoxide_ugm3 DOUBLE,
  avg_carbon_dioxide_ppm DOUBLE, max_carbon_dioxide_ppm DOUBLE,
  avg_pm2_5_ugm3 DOUBLE, max_pm2_5_ugm3 DOUBLE, avg_pm10_ugm3 DOUBLE, avg_nitrogen_dioxide_ugm3 DOUBLE,
  exceeds_who_pm2_5 BOOLEAN, exceeds_who_pm10 BOOLEAN, exceeds_who_no2 BOOLEAN, exceeds_who_co BOOLEAN,
  load_timestamp TIMESTAMP NOT NULL)
  CLUSTER BY (location_key, date_key)""")

spark.sql(f"CREATE TABLE IF NOT EXISTS {T_WM} (processed_up_to TIMESTAMP, updated_at TIMESTAMP)")

for stmt in [
    f"ALTER TABLE {T_LOC} ADD CONSTRAINT pk_dim_location PRIMARY KEY (location_key)",
    f"ALTER TABLE {T_DATE} ADD CONSTRAINT pk_dim_date PRIMARY KEY (date_key)",
    f"ALTER TABLE {T_FACT} ADD CONSTRAINT pk_fact_concentrations PRIMARY KEY (location_key, date_key)",
    f"ALTER TABLE {T_FACT} ADD CONSTRAINT fk_fact_location FOREIGN KEY (location_key) REFERENCES {T_LOC}(location_key)",
    f"ALTER TABLE {T_FACT} ADD CONSTRAINT fk_fact_date FOREIGN KEY (date_key) REFERENCES {T_DATE}(date_key)"]:
    try:
        spark.sql(stmt)
    except Exception as e:
        print("constraint skipped (already exists?):", str(e)[:90])
print("Gold tables ready in", GOLD)

In [0]:
# ==========================================
# Builders + MERGE helper
# ==========================================
def loc_key(c):
    """Deterministic surrogate key from the city name (same value in dim and fact, no lookup join)."""
    return (F.abs(F.xxhash64(c)) % 1000000000).cast("int")

def date_key(c):
    return F.date_format(c, "yyyyMMdd").cast("int")

def merge_table(table, source_df, match_cond, change_cond=None):
    """MERGE into a UC table; returns (inserted, updated) from Delta's own metrics."""
    dt = DeltaTable.forName(spark, table)
    v0 = dt.history(1).select("version").first()[0]
    m = dt.alias("t").merge(source_df.alias("s"), match_cond)
    if change_cond:
        m = m.whenMatchedUpdateAll(condition=change_cond)
    m.whenNotMatchedInsertAll().execute()
    h = DeltaTable.forName(spark, table).history(1).select("version", "operationMetrics").first()
    if h["version"] == v0:
        return 0, 0
    mt = h["operationMetrics"]
    return int(mt.get("numTargetRowsInserted", 0)), int(mt.get("numTargetRowsUpdated", 0))

def changed_cond(cols):
    return " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in cols)

LOC_VALUES  = ["country", "region", "latitude", "longitude", "elevation", "timezone", "hemisphere", "climate_band"]
FACT_MEASURES = ["avg_carbon_monoxide_ugm3", "max_carbon_monoxide_ugm3", "avg_carbon_dioxide_ppm",
                 "max_carbon_dioxide_ppm", "avg_pm2_5_ugm3", "max_pm2_5_ugm3", "avg_pm10_ugm3",
                 "avg_nitrogen_dioxide_ugm3"]
FACT_FLAGS = ["exceeds_who_pm2_5", "exceeds_who_pm10", "exceeds_who_no2", "exceeds_who_co"]
FACT_VALUES = ["hours_measured", "is_complete_day"] + FACT_MEASURES + FACT_FLAGS


def build_dim_location(changed, load_ts):
    return (changed.groupBy("city").agg(
                F.max("latitude").alias("latitude"), F.max("longitude").alias("longitude"),
                F.max("elevation").alias("elevation"), F.max("timezone").alias("timezone"))
            .join(LOCATION_REF, "city", "left")
            .select(loc_key(col("city")).alias("location_key"), "city",
                    F.coalesce("country", F.lit("Unknown")).alias("country"),
                    F.coalesce("region", F.lit("Unknown")).alias("region"),
                    "latitude", "longitude", "elevation", "timezone",
                    F.when(col("latitude") >= 0, "Northern").otherwise("Southern").alias("hemisphere"),
                    F.when(F.abs("latitude") < 23.5, "Tropical").when(F.abs("latitude") < 35, "Subtropical")
                     .when(F.abs("latitude") < 55, "Temperate").otherwise("Subpolar").alias("climate_band"),
                    F.lit(load_ts).alias("load_timestamp")))


def build_dim_date(affected, load_ts):
    d = col("full_date")
    return (affected.select(col("d").alias("full_date")).distinct()
            .select(date_key(d).alias("date_key"), "full_date",
                    F.year(d).alias("year"), F.quarter(d).alias("quarter"), F.month(d).alias("month"),
                    F.date_format(d, "MMMM").alias("month_name"), F.dayofmonth(d).alias("day"),
                    F.dayofweek(d).alias("day_of_week"), F.date_format(d, "EEEE").alias("day_name"),
                    F.dayofweek(d).isin(1, 7).alias("is_weekend"),
                    F.when(F.month(d).isin(12, 1, 2), "Winter").when(F.month(d).isin(3, 4, 5), "Spring")
                     .when(F.month(d).isin(6, 7, 8), "Summer").otherwise("Autumn").alias("season_northern_hemisphere"),
                    F.lit(load_ts).alias("load_timestamp")))


def build_fact(silver, affected, load_ts):
    """Re-aggregate the FULL day from Silver for every affected city-day (hourly -> daily)."""
    day = silver.withColumn("d", F.to_date("reading_time"))
    agg = (day.join(affected, ["city", "d"]).groupBy("city", "d").agg(
        F.count("*").cast("int").alias("hours_measured"),
        F.round(F.avg("carbon_monoxide_ugm3"), 4).alias("avg_carbon_monoxide_ugm3"),
        F.max("carbon_monoxide_ugm3").alias("max_carbon_monoxide_ugm3"),
        F.round(F.avg("carbon_dioxide_ppm"), 4).alias("avg_carbon_dioxide_ppm"),
        F.max("carbon_dioxide_ppm").alias("max_carbon_dioxide_ppm"),
        F.round(F.avg("pm2_5_ugm3"), 4).alias("avg_pm2_5_ugm3"),
        F.max("pm2_5_ugm3").alias("max_pm2_5_ugm3"),
        F.round(F.avg("pm10_ugm3"), 4).alias("avg_pm10_ugm3"),
        F.round(F.avg("nitrogen_dioxide_ugm3"), 4).alias("avg_nitrogen_dioxide_ugm3")))
    return agg.select(
        loc_key(col("city")).alias("location_key"), date_key(col("d")).alias("date_key"),
        "hours_measured", (col("hours_measured") >= 23).alias("is_complete_day"),   # tolerates a DST day
        *FACT_MEASURES,
        (col("avg_pm2_5_ugm3") > WHO_24H["pm2_5"]).alias("exceeds_who_pm2_5"),
        (col("avg_pm10_ugm3") > WHO_24H["pm10"]).alias("exceeds_who_pm10"),
        (col("avg_nitrogen_dioxide_ugm3") > WHO_24H["no2"]).alias("exceeds_who_no2"),
        (col("avg_carbon_monoxide_ugm3") > WHO_24H["co"]).alias("exceeds_who_co"),
        F.lit(load_ts).alias("load_timestamp"))


def publish_audit_tables():
    """Never raises (an audit copy must not break a Gold run). If a quarantine source is missing
    an EMPTY table with the right columns is published instead."""
    sources = [(log_table_path, "audit_pipeline_execution_logs", None),
               (quarantine_path, "audit_bronze_quarantine", bronze_quarantine_schema),
               (silver_quarantine_path, "audit_silver_quarantine", None)]
    for src, name, empty_schema in sources:
        try:
            try:
                df = spark.read.format("delta").load(src)
                df.columns                      # forces the read to resolve NOW
            except Exception as e:
                if empty_schema is None:
                    raise
                print(f"{name}: source not found, publishing an empty table")
                df = spark.createDataFrame([], empty_schema)
            (df.write.mode("overwrite").option("overwriteSchema", "true")
               .saveAsTable(f"{GOLD}.{name}"))
        except Exception as e:
            print(f"audit copy skipped for {name}: {str(e)[:200]}")

In [0]:
# ==========================================
# Silver -> Gold refresh
#   incremental : Silver rows that changed since the last Gold run (Silver.load_timestamp watermark)
#                 -> find the affected city-days -> re-aggregate them -> MERGE
#   backfill    : every city-day inside start_date..end_date (ignores the watermark)
# Idempotent: MERGE only touches rows whose values differ; re-running changes nothing.
# One audit-log row per Gold table (rows inserted/updated, run time).
# ==========================================
def refresh_gold(start_date=None, end_date=None, load_type="incremental"):
    layer = "Silver-to-Gold"
    t0 = datetime.now()
    status, error, detail = "Failure", None, "not started"
    counts = {"dim_location": (0, 0), "dim_date": (0, 0), "fact_concentrations": (0, 0)}
    print(f"--- {layer} [{load_type}] ---")

    try:
        load_ts = datetime.now()
        silver = spark.read.format("delta").load(silver_delta_path)
        new_wm = None
        if start_date and end_date:
            changed = silver.filter(F.to_date("reading_time").between(start_date, end_date))
            detail = f"reading dates {start_date} to {end_date}"
        else:
            wm = spark.table(T_WM).agg(F.max("processed_up_to")).first()[0]
            changed = silver.filter(col("load_timestamp") > F.lit(wm)) if wm else silver
            new_wm = changed.agg(F.max("load_timestamp")).first()[0]
            detail = f"silver load_timestamp in ({wm}, {new_wm}]"
            if new_wm is None:
                status, detail = "Success", detail + " | no new Silver changes"
                print("No new Silver data since the last Gold run.")
                return status
            changed = changed.filter(col("load_timestamp") <= F.lit(new_wm))

        affected = changed.select("city", F.to_date("reading_time").alias("d")).distinct()

        counts["dim_location"] = merge_table(T_LOC, build_dim_location(changed, load_ts),
                                             "t.location_key = s.location_key", changed_cond(LOC_VALUES))
        counts["dim_date"] = merge_table(T_DATE, build_dim_date(affected, load_ts),
                                         "t.date_key = s.date_key")          # insert-only
        fact_src = build_fact(silver, affected, load_ts)
        stats = fact_src.agg(F.count("*").alias("city_days"), F.sum("hours_measured").alias("silver_rows")).first()
        detail += f" | silver rows aggregated: {stats['silver_rows']} -> {stats['city_days']} city-days"
        counts["fact_concentrations"] = merge_table(
            T_FACT, fact_src, "t.location_key = s.location_key AND t.date_key = s.date_key",
            changed_cond(FACT_VALUES))

        if new_wm is not None:
            spark.createDataFrame([(new_wm, datetime.now())], "processed_up_to timestamp, updated_at timestamp") \
                 .write.mode("append").saveAsTable(T_WM)
        if sum(counts["fact_concentrations"]) > 0:
            spark.sql(f"OPTIMIZE {T_FACT}")                                  # liquid clustering
        status = "Success"
        print(f"Success: {counts}")

    except Exception as e:
        error = str(e)[:1000]
        print(f"FAILED: {error}")

    finally:
        for table, (ins, upd) in counts.items():
            write_log(layer, f"{table} | {detail}", load_type, t0, status,
                      rows_inserted=ins, rows_updated=upd, rows_quarantined=0, error_message=error)
        publish_audit_tables()
    return status

In [0]:
# ==========================================
# RUN SETTINGS: edit the values, then run the next cell
#   MODE        "incremental" -> Silver rows that changed since the last Gold run (no dates)
#               "backfill"    -> re-aggregate every city-day in a reading-date range
#   START_DATE  backfill only, "YYYY-MM-DD" (END_DATE defaults to START_DATE if left empty)
#   END_DATE    backfill only, "YYYY-MM-DD"
# ==========================================
MODE = "backfill"
START_DATE = "2023-10-02"
END_DATE = ""

In [0]:
settings = validate_run_settings(
    get_run_settings({"mode": MODE, "start_date": START_DATE, "end_date": END_DATE}), layer="gold")

refresh_gold(settings["start_date"] or None, settings["end_date"] or None, load_type=settings["mode"])

In [0]:
if "PIPELINE_SETTINGS" not in globals():   # skipped when run from 04_Run_Pipeline
    # ==========================================
    # Evidence: Gold tables, a star-schema query, optimisation details, audit log
    # ==========================================
    for t in [T_LOC, T_DATE, T_FACT]:
        print(f"{t}: {spark.table(t).count():,} rows")

    print("Business question: average CO2 concentration per region per year (complete days only)")
    display(spark.sql(f"""
        SELECT l.region, d.year, ROUND(AVG(f.avg_carbon_dioxide_ppm), 1) AS avg_co2_ppm,
               COUNT(f.avg_carbon_dioxide_ppm) AS days_with_co2, COUNT(*) AS days
        FROM {T_FACT} f JOIN {T_LOC} l USING (location_key) JOIN {T_DATE} d USING (date_key)
        WHERE f.is_complete_day GROUP BY l.region, d.year ORDER BY l.region, d.year"""))

    print("Read optimisation: liquid clustering keys and file layout of the fact table")
    display(spark.sql(f"DESCRIBE DETAIL {T_FACT}").select("clusteringColumns", "numFiles", "sizeInBytes"))

    display(spark.read.format("delta").load(log_table_path)
            .filter(col("layer_processed") == "Silver-to-Gold").orderBy(col("start_time").desc()).limit(9))

In [0]:
if "PIPELINE_SETTINGS" not in globals():   # skipped when run from 04_Run_Pipeline
    # ==========================================
    # PROOF: Gold is idempotent. Re-running the same window changes nothing
    # (expect rows_inserted = 0 and rows_updated = 0 for all three tables on the second run).
    # ==========================================
    from datetime import timedelta
    latest = spark.table(T_DATE).agg(F.max("full_date")).first()[0]
    w_start, w_end = str(latest - timedelta(days=2)), str(latest)
    refresh_gold(w_start, w_end, load_type="rerun_test")
    refresh_gold(w_start, w_end, load_type="rerun_test")
    display(spark.read.format("delta").load(log_table_path)
            .filter((col("layer_processed") == "Silver-to-Gold") & (col("load_type") == "rerun_test"))
            .orderBy(col("start_time").desc()).limit(6))